# Basira — train a Kraken Arabic **handwriting** model (Muharaf)

**Before running:** Runtime → Change runtime type → **GPU (T4)**.

Transfer-learns from the OpenITI Arabic base model on the `aamijar/muharaf-public`
line dataset (24,495 line image↔text pairs). Output: checkpoints in **`muharaf_hw/`**.
~1–3 h on a free T4. (Tested against **Kraken 7.x** CLI.)

⚠️ **Licence:** Muharaf is **CC BY-NC-SA (non-commercial)** — R&D / demo use only.
It installs into Basira's demo-gated slot.


In [ ]:
# 1) Install Kraken + check the GPU
!pip install -q kraken datasets pillow
import torch
print("CUDA available:", torch.cuda.is_available())   # must print True
# If pip says "restart": Runtime > Restart session, then re-run from here.


In [ ]:
# 2) Build Kraken training pairs (line image + .gt.txt) from the dataset
import os
from datasets import load_dataset

ds = load_dataset("aamijar/muharaf-public")           # public; no token needed
# QUICK TEST first? Uncomment to train on a small subset (~20 min):
# ds["train"] = ds["train"].select(range(2000))

for split in ds:                                       # train / validation / test
    out = f"data/{split}"; os.makedirs(out, exist_ok=True); n = 0
    for i, row in enumerate(ds[split]):
        text = (row.get("text") or "").strip()
        if not text:
            continue
        img = row["image"]
        if img.mode not in ("RGB", "L"):
            img = img.convert("RGB")
        img.save(f"{out}/{i:06d}.png")
        with open(f"{out}/{i:06d}.gt.txt", "w", encoding="utf-8") as fh:
            fh.write(text)
        n += 1
    print(split, "->", n, "line pairs")


In [ ]:
# 3) Fetch the OpenITI Arabic base model (for transfer learning)
!kraken get 10.5281/zenodo.7050270
import glob, os
base = glob.glob(os.path.expanduser("~/.local/share/htrmopo/**/*.mlmodel"), recursive=True)[0]
print("base model:", base)


In [ ]:
# 4) Train. NOTE (Kraken 7): -d/--device is a `ketos` group option (BEFORE
# `train`), and -o is a DIRECTORY — checkpoints are written into muharaf_hw/.
import glob, os
base = glob.glob(os.path.expanduser("~/.local/share/htrmopo/**/*.mlmodel"), recursive=True)[0]
cmd = f'ketos -d cuda:0 train -f path --resize union -i "{base}" -o muharaf_hw data/train/*.png'
print(cmd)
!{cmd}


In [ ]:
# 5) Evaluate on the held-out test split (character accuracy / CER)
import glob
ckpts = sorted(glob.glob("muharaf_hw/*.mlmodel"))
print("checkpoints:", ckpts)
model = ([c for c in ckpts if "best" in c] or ckpts)[-1]
print("evaluating:", model)
!ketos -d cuda:0 test -f path -m "{model}" data/test/*.png


In [ ]:
# 6) Save the model to Google Drive (reliable from VS Code or browser)
import glob
ckpts = sorted(glob.glob("muharaf_hw/*.mlmodel"))
best = ([c for c in ckpts if "best" in c] or ckpts)[-1]
print("model:", best)
from google.colab import drive
drive.mount("/content/drive")
!cp "{best}" /content/drive/MyDrive/muharaf_hw_best.mlmodel
print("Saved to Drive as muharaf_hw_best.mlmodel — download it from drive.google.com")


## Back on your Mac
Download `muharaf_hw_best.mlmodel` from Google Drive, then with the app + Kraken
sidecar running:
```bash
scripts/install-kraken-model.sh ~/Downloads/muharaf_hw_best.mlmodel muharaf
```
Set `DEMO_TRANSCRIBE_ADAPTER=kraken-muharaf` in `.env`, restart (`pnpm dev`),
log in as `demo@basira.test`, and transcribe — it runs your handwriting model.
